
# Telco Customer Churn — Feature Engineering & Preprocessing

## Objective

This notebook prepares the Telco Customer Churn dataset for machine learning.

The main objectives are:
- Apply consistent data cleaning.
- Separate numerical and categorical features.
- Encode categorical variables.
- Scale numerical features.
- Build reusable preprocessing pipelines.
- Prevent data leakage during cross-validation.


In [1]:

from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.base import clone

# Project root
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.preprocessing import (
    clean_telco_data,
    build_preprocessor
)



## 1. Data Preparation

The raw dataset is cleaned using the reusable
`clean_telco_data()` function.

The dataset is then split into:
- 80% training data
- 20% held-out test data

Stratification preserves the churn class distribution
across both subsets.

The customer identifier is excluded from the predictors,
and the target variable is encoded as:
- 0: No churn
- 1: Churn

The test set is reserved for final model evaluation.

All preprocessing operations that learn parameters
from data will be fitted exclusively on the training
folds during cross-validation to prevent data leakage.


In [2]:
data_path = (
    project_root / "data" / "raw"
    / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

raw_df = pd.read_csv(data_path)

df = clean_telco_data(raw_df)

print("Dataset shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("TotalCharges dtype:", df["TotalCharges"].dtype)


Dataset shape: (7043, 21)
Missing values: 0
TotalCharges dtype: float64


In [3]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Churn"]
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain churn distribution:")
print(train_df["Churn"].value_counts(normalize=True))

print("\nTest churn distribution:")
print(test_df["Churn"].value_counts(normalize=True))


Train shape: (5634, 21)
Test shape: (1409, 21)

Train churn distribution:
Churn
No     0.734647
Yes    0.265353
Name: proportion, dtype: float64

Test churn distribution:
Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64


In [4]:
# Features
X_train = train_df.drop(
    columns=["customerID", "Churn"]
)

X_test = test_df.drop(
    columns=["customerID", "Churn"]
)

# Binary target
y_train = train_df["Churn"].map({
    "No": 0,
    "Yes": 1
})

y_test = test_df["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (5634, 19)
X_test: (1409, 19)
y_train: (5634,)
y_test: (1409,)



## 2. Feature Types and Encoding Strategy

The 19 predictors are divided into three groups:

- Numerical features (3): standardized for linear models
  and preserved on their original scale for tree-based models.
- Binary numerical features (1): preserved without scaling.
- Categorical features (15): encoded using OneHotEncoder.

The same feature groups are used in both preprocessing
configurations to ensure consistent model comparisons.

All data-dependent transformations will be fitted within
the training folds during cross-validation to prevent
data leakage.

In [5]:
# Numerical features
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

# Binary numerical features
binary_features = [
    "SeniorCitizen"
]

# Categorical features
categorical_features = [
    col for col in X_train.columns
    if col not in numeric_features + binary_features
]

print("Numerical features:", numeric_features)
print("\nBinary features:", binary_features)
print("\nCategorical features:", categorical_features)

assert set(
    numeric_features + binary_features + categorical_features
) == set(X_train.columns)

assert len(
    numeric_features + binary_features + categorical_features
) == X_train.shape[1]


Numerical features: ['tenure', 'MonthlyCharges', 'TotalCharges']

Binary features: ['SeniorCitizen']

Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [6]:
# Inspect categorical features
for col in categorical_features:
    print(f"{col}: {X_train[col].unique().tolist()}")

gender: ['Male', 'Female']
Partner: ['No', 'Yes']
Dependents: ['No', 'Yes']
PhoneService: ['No', 'Yes']
MultipleLines: ['No phone service', 'No', 'Yes']
InternetService: ['DSL', 'Fiber optic', 'No']
OnlineSecurity: ['No', 'Yes', 'No internet service']
OnlineBackup: ['No', 'Yes', 'No internet service']
DeviceProtection: ['Yes', 'No', 'No internet service']
TechSupport: ['No', 'Yes', 'No internet service']
StreamingTV: ['Yes', 'No', 'No internet service']
StreamingMovies: ['Yes', 'No', 'No internet service']
Contract: ['Month-to-month', 'Two year', 'One year']
PaperlessBilling: ['No', 'Yes']
PaymentMethod: ['Electronic check', 'Mailed check', 'Credit card (automatic)', 'Bank transfer (automatic)']



## 3. Preprocessing Pipelines

Two preprocessing strategies are implemented to accommodate
different machine learning algorithms.

**Linear models (Logistic Regression):**
- Median imputation for numerical features.
- StandardScaler for numerical features.
- Most-frequent imputation for categorical features.
- OneHotEncoder for categorical features.
- Binary numerical features are preserved.

**Tree-based models (Decision Tree, Random Forest):**
- The same imputation and encoding strategy.
- Numerical features retain their original scale.
- Binary numerical features are preserved.

Both configurations use ColumnTransformer and will be
integrated into scikit-learn Pipelines during model training
to prevent data leakage.


In [7]:
# Logistic Regression preprocessing
preprocessor_linear = build_preprocessor(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    binary_features=binary_features,
    scale_numeric=True
)

# Random Forest preprocessing
preprocessor_tree = build_preprocessor(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    binary_features=binary_features,
    scale_numeric=False
)

print("Both preprocessors created successfully.")


Both preprocessors created successfully.


In [8]:
fitted_linear = clone(preprocessor_linear)
X_train_linear = fitted_linear.fit_transform(X_train)

# Get transformed feature names
feature_names = fitted_linear.get_feature_names_out()

# Convert to DataFrame
df_linear = pd.DataFrame(
    X_train_linear,
    columns=feature_names,
    index=X_train.index
)

display(df_linear.head(10))


,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,...,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen
3738,0.102371,-0.521976,-0.262257,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3151,-0.711743,0.337478,-0.503635,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4860,-0.793155,-0.809013,-0.749883,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3867,-0.263980,0.284384,-0.172722,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3810,-1.281624,-0.676279,-0.989374,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2666,1.364249,0.483485,1.271067,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2645,-1.281624,-1.497571,-1.000234,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
189,0.305900,-0.276418,0.035000,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
5767,1.323543,-0.170230,0.662073,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2257,1.120015,0.518328,1.117908,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [9]:
fitted_tree = clone(preprocessor_tree)
X_train_tree = fitted_tree.fit_transform(X_train)

df_tree = pd.DataFrame(
    X_train_tree,
    columns=fitted_tree.get_feature_names_out(),
    index=X_train.index
)

display(df_tree.head(10))

,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,...,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen
3738,35.0,49.20,1701.65,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3151,15.0,75.10,1151.55,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4860,13.0,40.55,590.35,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3867,26.0,73.50,1905.70,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3810,1.0,44.55,44.55,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2666,66.0,79.50,5196.10,0.0,1.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2645,1.0,19.80,19.80,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
189,40.0,56.60,2379.10,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
5767,65.0,59.80,3808.20,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2257,60.0,80.55,4847.05,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [10]:
display(
    pd.DataFrame({
        "Original": X_train["MonthlyCharges"].head(5).to_numpy(),
        "Logistic Regression": df_linear["MonthlyCharges"].head(5).to_numpy(),
        "Random Forest": df_tree["MonthlyCharges"].head(5).to_numpy()
    })
)

,Original,Logistic Regression,Random Forest
0,49.20,-0.521976,49.20
1,75.10,0.337478,75.10
2,40.55,-0.809013,40.55
3,73.50,0.284384,73.50
4,44.55,-0.676279,44.55



## 4. Preprocessing Validation

Before model training, we validate the preprocessing
pipelines to ensure that:

- The expected number of features is generated.
- No missing or infinite values remain.
- Numerical features are standardized for linear models.
- Numerical features retain their original scale for tree-based models.
- Categorical encoding is consistent across pipelines.
- The number and order of observations are preserved.

These checks help ensure that the transformed features
are suitable for machine learning.


In [11]:
encoder = (
    fitted_linear
    .named_transformers_["cat"]
    .named_steps["encoder"]
)

n_encoded = sum(len(categories) for categories in encoder.categories_)

expected_features = (
    len(numeric_features)
    + n_encoded
    + len(binary_features)
)

assert df_linear.shape == (len(X_train), expected_features)
assert df_tree.shape == (len(X_train), expected_features)

assert df_linear.index.equals(X_train.index)
assert df_tree.index.equals(X_train.index)

assert np.isfinite(df_linear.to_numpy()).all()
assert np.isfinite(df_tree.to_numpy()).all()

print("Original features:", X_train.shape[1])
print("Encoded categorical features:", n_encoded)
print("Transformed features:", expected_features)
print("All dimension and missing-value checks passed.")


Original features: 19
Encoded categorical features: 41
Transformed features: 45
All dimension and missing-value checks passed.


In [12]:
np.testing.assert_allclose(
    df_linear[numeric_features].mean().to_numpy(),
    np.zeros(len(numeric_features)),
    atol=1e-10
)

np.testing.assert_allclose(
    df_linear[numeric_features].std(ddof=0).to_numpy(),
    np.ones(len(numeric_features)),
    atol=1e-10
)

# Tree pipeline: original numerical values preserved
np.testing.assert_allclose(
    df_tree[numeric_features].to_numpy(),
    X_train[numeric_features].to_numpy()
)

# Binary feature preserved in both pipelines
for transformed_df in [df_linear, df_tree]:
    np.testing.assert_array_equal(
        transformed_df["SeniorCitizen"].to_numpy(),
        X_train["SeniorCitizen"].to_numpy()
    )

print("Numerical preprocessing checks passed.")


Numerical preprocessing checks passed.


In [13]:
encoded_features = encoder.get_feature_names_out(
    categorical_features
)

np.testing.assert_array_equal(
    df_linear[encoded_features].to_numpy(),
    df_tree[encoded_features].to_numpy()
)

for feature, categories in zip(
    categorical_features,
    encoder.categories_
):
    encoded_columns = [
        f"{feature}_{category}"
        for category in categories
    ]

    row_sums = df_linear[encoded_columns].sum(axis=1)

    assert (row_sums == 1).all(), (
        f"Invalid one-hot encoding for {feature}"
    )

print("Categorical encoding checks passed.")


Categorical encoding checks passed.


In [14]:
preprocessing_summary = pd.DataFrame({
    "Feature type": [
        "Numerical",
        "Categorical",
        "Binary numerical",
        "Total"
    ],
    "Original features": [
        len(numeric_features),
        len(categorical_features),
        len(binary_features),
        X_train.shape[1]
    ],
    "Transformed features": [
        len(numeric_features),
        n_encoded,
        len(binary_features),
        expected_features
    ]
})

display(preprocessing_summary)


,Feature type,Original features,Transformed features
0,Numerical,3,3
1,Categorical,15,41
2,Binary numerical,1,1
3,Total,19,45



## 5. Conclusion

### Data Preparation

- The dataset contains 7,043 customers and 21 columns.
- Invalid `TotalCharges` values were corrected using the data cleaning function.
- The data was split into stratified training (80%) and test (20%) sets.
- `customerID` was excluded from the predictors.
- The target variable `Churn` was encoded as a binary variable.

### Feature Engineering

The original 19 predictors were transformed into 45 features:

- 3 numerical features.
- 41 one-hot encoded categorical features.
- 1 binary numerical feature.

Two reusable preprocessing configurations were implemented:

**Linear preprocessing**
- Numerical imputation and standardization.
- Categorical imputation and one-hot encoding.
- Binary feature passthrough.

**Tree-based preprocessing**
- Numerical imputation without standardization.
- Categorical imputation and one-hot encoding.
- Binary feature passthrough.

### Validation

The preprocessing validation checks confirm:

- Expected output dimensions.
- No missing or infinite values.
- Correct numerical transformations.
- Consistent categorical encoding.
- Preserved observation order and count.

### Data Leakage Prevention

All diagnostic transformations were fitted exclusively
on the training set.

During model training, preprocessing will be integrated
into complete scikit-learn Pipelines and fitted separately
within each cross-validation fold.

The held-out test set remains reserved for final evaluation.
